In [5]:
import sys
sys.path.append('../src')

import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

from model import SimpleCNN
from partition import dirichlet_partition
from federated import local_train, federated_average, evaluate
from attack import get_confidence_and_loss, build_attack_dataset, run_mia_attack

transform = transforms.ToTensor()
train_dataset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform)
train_labels = [label for _, label in train_dataset]

num_clients = 5
alpha = 0.5
client_data = dirichlet_partition(train_labels, num_clients=num_clients, alpha=alpha)

In [6]:
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

global_model = SimpleCNN(num_classes=10)
num_rounds = 5

for round_num in range(num_rounds):
    client_state_dicts = []
    
    for client_id in range(num_clients):
        subset = Subset(train_dataset, client_data[client_id])
        dataloader = DataLoader(subset, batch_size=32, shuffle=True)
        state_dict = local_train(global_model, dataloader, epochs=1, lr=0.01)
        client_state_dicts.append(state_dict)
    
    new_global_state = federated_average(client_state_dicts)
    global_model.load_state_dict(new_global_state)
    
    acc = evaluate(global_model, test_loader)
    print(f"Round {round_num+1}/{num_rounds} - Global model test accuracy: {acc:.4f}")

Round 1/5 - Global model test accuracy: 0.1186
Round 2/5 - Global model test accuracy: 0.1911
Round 3/5 - Global model test accuracy: 0.2259
Round 4/5 - Global model test accuracy: 0.2622
Round 5/5 - Global model test accuracy: 0.2904


In [7]:
# "member" samples: client 0's actual training data
member_subset = Subset(train_dataset, client_data[0])
member_loader = DataLoader(member_subset, batch_size=64, shuffle=False)

# "non-member" samples: the test set (never used in training)
nonmember_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

member_conf, member_loss = get_confidence_and_loss(global_model, member_loader)
nonmember_conf, nonmember_loss = get_confidence_and_loss(global_model, nonmember_loader)

X, y = build_attack_dataset(member_conf, member_loss, nonmember_conf, nonmember_loss)
metrics = run_mia_attack(X, y)

print("MIA Attack Results:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

MIA Attack Results:
accuracy: 0.5310
precision: 0.5289
recall: 0.7511
f1: 0.6207
